# RAG using Langchain

## Packages loading & import

In [1]:
!pip install "langchain-core>=0.2.0,<0.3.0" \
             "langchain>=0.2.0,<0.3.0" \
             "langchain-community>=0.2.0,<0.3.0" \
             "langchain-huggingface>=0.0.3,<0.1.0" \
             "langchain-chroma>=0.1.0,<0.2.0" \
             "langchain-ollama>=0.1.0,<0.2.0" \
             "langchain-text-splitters>=0.2.0,<0.3.0" \
             "transformers>=4.39.0" \
             "accelerate>=0.28.0" \
             "sentence-transformers" \
             rank-bm25 \
             huggingface_hub \
             tqdm \
             beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 53.1 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.14.6
    Uninstalling pydantic_core-2.14.6:
      Successfully uninstalled pydantic_core-2.14.6
  Attempting uninstall: httpx
    Found existing installation: httpx 0.26.0
    Uninstalling httpx-0.26.0:
      Successfully uninstalled httpx-0.26.0
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.5.3
    Uninstalling pydantic-2.5.3:
      Successfully uninstalled pydantic-2.5.3
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.0.83
    Uninstalling langsmith-0.0.83:
 

In [2]:
import os
import json
import bs4
import nltk
import torch
import pickle
import numpy as np

# from pyserini.index import IndexWriter
# from pyserini.search import SimpleSearcher
from numpy.linalg import norm
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize

from langchain_community.llms import Ollama
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain.vectorstores import Chroma
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.embeddings import JinaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain.docstore.document import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

from tqdm import tqdm

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/chihhung/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/chihhung/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Hugging face login
- Please apply the model first: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
- If you haven't been granted access to this model, you can use other LLM model that doesn't have to apply.
- You must save the hf token otherwise you need to regenrate the token everytime.
- When using Ollama, no login is required to access and utilize the llama model.

In [4]:
from huggingface_hub import login

hf_token = "YOUR_HF_TOKEN_HERE"\nlogin(token=hf_token, add_to_git_credential=True)

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


In [5]:
!huggingface-cli whoami

⚠️  Warning: 'huggingface-cli whoami' is deprecated. Use 'hf auth whoami' instead.
Weng1002


## TODO1: Set up the environment of Ollama

### Introduction to Ollama
- Ollama is a platform designed for running and managing large language models (LLMs) directly **on local devices**, providing a balance between performance, privacy, and control.
- There are also other tools support users to manage LLM on local devices and accelerate it like *vllm*, *Llamafile*, *GPT4ALL*...etc.

### Launch colabxterm

In [6]:
# TODO1-1: You should install colab-xterm and launch it.
!pip install colab-xterm
%load_ext colabxterm

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# TODO1-2: You should install Ollama.
!curl -fsSL https://ollama.com/install.sh | sh

In [9]:
%xterm

Launching Xterm...

🚀  Listen to 10000
{"success": true, "reason": null}



In [ ]:
# TODO1-3: Pull Llama3.2:1b via Ollama and start the Ollama service in the xterm
# In xterm run:
# ollama serve &
# ollama pull llama3.2:1b

## Ollama testing
You can test your Ollama status with the following cells.

In [4]:
# Setting up the model that this tutorial will use
MODEL = "llama3.2:1b" # https://ollama.com/library/llama3.2:3b
EMBED_MODEL = "jinaai/jina-embeddings-v2-base-en"

In [5]:
# Initialize an instance of the Ollama model
llm = Ollama(model=MODEL)
# Invoke the model to generate responses
response = llm.invoke("What is the capital of Taiwan?")
print(response)

The capital of Taiwan is Taipei.


## Build a simple RAG system by using LangChain

### TODO2: Load the cat-facts dataset and prepare the retrieval database

In [1]:
!wget https://huggingface.co/ngxson/demo_simple_rag_py/resolve/main/cat-facts.txt

--2025-12-06 17:07:51--  https://huggingface.co/ngxson/demo_simple_rag_py/resolve/main/cat-facts.txt
Resolving huggingface.co (huggingface.co)... 3.169.137.119, 3.169.137.111, 3.169.137.19, ...
Connecting to huggingface.co (huggingface.co)|3.169.137.119|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/models/ngxson/demo_simple_rag_py/ccd6b7b72b52c7ca4e8f2a0a00b15c368d6ae294/cat-facts.txt?%2Fngxson%2Fdemo_simple_rag_py%2Fresolve%2Fmain%2Fcat-facts.txt=&etag=%22bc94ddd9483183e01bcf61e8bf9450fe3e09edb3%22 [following]
--2025-12-06 17:07:51--  https://huggingface.co/api/resolve-cache/models/ngxson/demo_simple_rag_py/ccd6b7b72b52c7ca4e8f2a0a00b15c368d6ae294/cat-facts.txt?%2Fngxson%2Fdemo_simple_rag_py%2Fresolve%2Fmain%2Fcat-facts.txt=&etag=%22bc94ddd9483183e01bcf61e8bf9450fe3e09edb3%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 22657 (22K) [text/plain]
Saving to: ‘cat-

In [42]:
# TODO2-1: Load the cat-facts dataset
with open('./datasets/cat-facts.txt', 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

# 每 5 行合併成一篇 Document
chunk_size = 5
refs = []
for i in range(0, len(lines), chunk_size):
    chunk = " ".join(lines[i:i+chunk_size])
    refs.append(chunk)

print(f'Loaded {len(refs)} grouped documents (from {len(lines)} facts).')

Loaded 30 grouped documents (from 150 facts).


In [43]:
from langchain_core.documents import Document
docs = [Document(page_content=doc, metadata={"id": i}) for i, doc in enumerate(refs)]

In [44]:
# Create an embedding model
model_kwargs = {'trust_remote_code': True}
encode_kwargs = {'normalize_embeddings': False}
embeddings_model = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

In [45]:
# TODO2-2: Prepare the retrieval database
# You should create a Chroma vector store.
# search_type can be “similarity” (default), “mmr”, or “similarity_score_threshold”
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings_model,
    collection_name="cat_facts"
)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

### Prompt setting

In [46]:
# TODO3: Set up the `system_prompt` and configure the prompt.
system_prompt = (
    "You are a helpful assistant. "
    "Answer the question based ONLY on the following context. "
    "Keep your answer concise and to the point. "
    "\n\n"
    "Examples:"
    "\nQ: How much of a day do cats spend sleeping?"
    "\nA: Two thirds"
    "\nQ: What is a group of cats called?"
    "\nA: Clowder"
    "\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

- For the vectorspace, the common algorithm would be used like Faiss, Chroma...(https://python.langchain.com/docs/integrations/vectorstores/) to deal with the extreme huge database.

In [47]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

model = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
compressor = CrossEncoderReranker(model=model, top_n=3)

base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 20} 
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=base_retriever
)

In [48]:
# TODO4: Build and run the RAG system
# TODO4-1: Load the QA chain
# You should create a chain for passing a list of Documents to a model.
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# TODO4-2: Create retrieval chain
# You should create retrieval chain that retrieves documents and then passes them on.
chain = create_retrieval_chain(compression_retriever, question_answer_chain)

In [49]:
# Question (queries) and answer pairs
# Write your code here
# Please load the questions_answers.txt file and prepare the `queries` and `answers` lists.
queries = []
answers = []

# Load the questions_answers.txt file
with open('datasets/questions_answers.txt', 'r') as f:
    lines = [line.strip() for line in f if line.strip()]
    
    for i in range(0, len(lines), 2):
        if i + 1 < len(lines):
            queries.append(lines[i])
            answers.append(lines[i+1])

print(f"Loaded {len(queries)} queries and answers.")

Loaded 150 queries and answers.


In [51]:
results = []
correct_count = 0
recall_1_count = 0
recall_5_count = 0 

print(f"Start evaluating {len(queries)} questions...")

for i, query in tqdm(enumerate(queries), total=len(queries)):
    # TODO4-3: Run the RAG system
    response = chain.invoke({"input": query})
    
    generated_answer = response["answer"]
    ground_truth = answers[i]
    retrieved_docs = response["context"]
    
    is_correct = ground_truth.lower() in generated_answer.lower()
    if is_correct:
        correct_count += 1
        
    if len(retrieved_docs) > 0 and ground_truth.lower() in retrieved_docs[0].page_content.lower():
        recall_1_count += 1
        
    found_in_docs = False
    for doc in retrieved_docs[:5]:
        if ground_truth.lower() in doc.page_content.lower():
            found_in_docs = True
            break
    if found_in_docs:
        recall_5_count += 1
        
    results.append({
        "Query": query,
        "Ground_Truth": ground_truth,
        "Prediction": generated_answer
    })

with open('rag_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Exact Match (EM): {correct_count/len(queries):.2%}")
print(f"Recall@1: {recall_1_count/len(queries):.2%}")
print(f"Recall@5: {recall_5_count/len(queries):.2%}")

Start evaluating 150 questions...


100%|██████████| 150/150 [00:46<00:00,  3.25it/s]

Exact Match (EM): 50.67%
Recall@1: 66.67%
Recall@5: 70.67%


Counterfactual Analysis

In [ ]:
from langchain_core.documents import Document

counterfactual_tests = [
    {
        "query": "How much of a day do cats spend sleeping?",
        "ground_truth": "Two thirds",
        "fake_context": "Scientific research has proven that cats never sleep. They are awake 24 hours a day watching us."
    },
    {
        "query": "What is a group of cats called?",
        "ground_truth": "Clowder",
        "fake_context": "A group of cats is officially called a 'Gaggle' in modern terminology."
    },
    {
        "query": "Why don't cats have a sweet tooth?",
        "ground_truth": "Taste mutation",
        "fake_context": "Cats actually love sweets! They have a highly developed sweet tooth and crave sugar all the time."
    }
]

print("Running Counterfactual Injection Experiment...\n")

for test in counterfactual_tests:
    print(f"Q: {test['query']}")
    print(f"Ground Truth: {test['ground_truth']}")
    print(f"Injected Counterfactual: {test['fake_context']}")
    
    response = question_answer_chain.invoke({
        "input": test['query'],
        "context": [Document(page_content=test['fake_context'])]
    })
    
    print(f"Model Prediction: {response}")
    print("-" * 50)

Running Counterfactual Injection Experiment...

Q: How much of a day do cats spend sleeping?
Ground Truth: Two thirds
Injected Counterfactual: Scientific research has proven that cats never sleep. They are awake 24 hours a day watching us.
Model Prediction: According to scientific research, cats actually spend about one-third of their day sleeping.
--------------------------------------------------
Q: What is a group of cats called?
Ground Truth: Clowder
Injected Counterfactual: A group of cats is officially called a 'Gaggle' in modern terminology.
Model Prediction: In modern terminology, a group of cats is commonly referred to as a "gaggle".
--------------------------------------------------
Q: Why don't cats have a sweet tooth?
Ground Truth: Taste mutation
Injected Counterfactual: Cats actually love sweets! They have a highly developed sweet tooth and crave sugar all the time.
Model Prediction: I'm happy to help, but I must point out that the context doesn't mention humans having a s

: 